# Trabalho Prático: Aprendizado de Máquina (Supervisionado e Não Supervisionado)
**Dataset:** QSAR Biodegradation (1054 amostras, 41 features moleculares)
**Objetivo:** Classificação binária de substâncias químicas (Prontamente Biodegradável - RB vs Não Prontamente Biodegradável - NRB).

## Estrutura do Notebook:
1. Configuração e Importação de Bibliotecas
2. Leitura, Limpeza e Normalização Obrigatória dos Dados
3. Divisão dos Dados (70% Treinamento / 15% Validação / 15% Teste)
4. Modelagem Supervisionada (Decision Tree, KNN, Artificial Neural Network) com otimização
5. Modelagem Não Supervisionada (K-Means) com Método do Cotovelo e Silhouette Score
6. Análise de Componentes Principais (PCA) para visualização dos Clusters

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Divisão
from sklearn.model_selection import train_test_split, PredefinedSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Modelos Supervisionados
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# Clusterização
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Métricas de Avaliação
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score, silhouette_score)

from imblearn.over_sampling import SMOTE

# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
import warnings
warnings.filterwarnings('ignore') # Ignorar avisos de depreciação menores

## 1. Carregamento, Limpeza e Normalização dos Dados
Mesmo que o arquivo original já contenha a palavra "normalizado", passaremos os dados por um pipeline rigoroso de verificação de nulos, separação de variáveis e padronização estatística (z-score). O resultado será exportado para um novo arquivo CSV limpo, cumprindo o requisito de ter um script de pré-processamento.

In [ ]:
# 1.1 Carregamento dos dados
# Lendo o arquivo original (certifique-se de que o arquivo está no mesmo diretório do notebook)
# Assumindo separador por vírgula com base nos metadados da amostra
df_raw = pd.read_csv('biodeg_normalizado.csv')

print(f"Dimensões originais do dataset: {df_raw.shape}")

# 1.2 Limpeza de Dados (Tratamento de Nulos/Duplicatas)
# Remoção de duplicatas se existirem
df_raw = df_raw.drop_duplicates()
# Remoção de valores nulos (se existirem)
df_raw = df_raw.dropna()
print(f"Dimensões após limpeza básica: {df_raw.shape}")

# 1.3 Separação de Features (X) e Target (y)
target_col = 'experimental class' # Conforme inspecionado no cabeçalho dos dados
X_raw = df_raw.drop(columns=[target_col])
y_raw = df_raw[target_col]

# 1.4 Codificação da Variável Alvo
# RB (Ready Biodegradable) e NRB (Not Ready Biodegradable)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
# Mapeamento para conferência: classes originais para valores numéricos
le_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(f"Mapeamento do Target: {le_mapping}")

# 1.5 Normalização Estatística (Z-Score)
# É crucial para o KNN, K-Means e Redes Neurais que as features estejam na mesma escala
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=X_raw.columns)

# 1.6 Salvando o Dataset Final Processado
df_clean = X_scaled.copy()
df_clean['target_encoded'] = y_encoded
df_clean.to_csv('biodeg_limpo_e_processado.csv', index=False)
print("\nDataset limpo e normalizado salvo com sucesso como 'biodeg_limpo_e_processado.csv'.")